In [ ]:
# Install newer, pinned versions of everything
!pip install --no-deps git+https://github.com/google/jax.git@8a04dfd830ff89f46e1fe3e866ee4fb2da9c90aa#egg=jax
!pip install --no-deps git+https://github.com/jax-ml/jax-triton.git@321a19e901ee318a43345485a47e711944011c77#egg=jax_triton
!pip install 'jaxlib[cuda12_pip] @ https://storage.googleapis.com/jax-releases/nightly/cuda12/jaxlib-0.4.16.dev20230831+cuda12.cudnn89-cp310-cp310-manylinux2014_x86_64.whl'
!pip install triton-nightly==2.1.0.dev20230722054455 --index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/Triton-Nightly/pypi/simple/

  Cloning https://github.com/google/jax.git (to revision 8a04dfd830ff89f46e1fe3e866ee4fb2da9c90aa) to /tmp/pip-install-w5lugkn4/jax_f2124dcb7f234116a86b9393a7852ddb
  Running command git clone --filter=blob:none --quiet https://github.com/google/jax.git /tmp/pip-install-w5lugkn4/jax_f2124dcb7f234116a86b9393a7852ddb
  Running command git rev-parse -q --verify 'sha^8a04dfd830ff89f46e1fe3e866ee4fb2da9c90aa'
  Running command git fetch -q https://github.com/google/jax.git 8a04dfd830ff89f46e1fe3e866ee4fb2da9c90aa
  Running command git checkout -q 8a04dfd830ff89f46e1fe3e866ee4fb2da9c90aa
  Resolved https://github.com/google/jax.git to commit 8a04dfd830ff89f46e1fe3e866ee4fb2da9c90aa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for jax: filename=jax-0.4.16.dev20230831-py3-none-any.whl size=1629823 sha256=5d6f6be7e671ceb00d2a19166403cea7dde894886470d70123ba91eb036a2324
  Stored in directory

  Cloning https://github.com/jax-ml/jax-triton.git (to revision 321a19e901ee318a43345485a47e711944011c77) to /tmp/pip-install-smf3ck5k/jax-triton_d9e38d8a22ce43249827ab0ef56814c8
  Running command git clone --filter=blob:none --quiet https://github.com/jax-ml/jax-triton.git /tmp/pip-install-smf3ck5k/jax-triton_d9e38d8a22ce43249827ab0ef56814c8
  Running command git rev-parse -q --verify 'sha^321a19e901ee318a43345485a47e711944011c77'
  Running command git fetch -q https://github.com/jax-ml/jax-triton.git 321a19e901ee318a43345485a47e711944011c77
  Running command git checkout -q 321a19e901ee318a43345485a47e711944011c77
  Resolved https://github.com/jax-ml/jax-triton.git to commit 321a19e901ee318a43345485a47e711944011c77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for jax_triton: filename=jax_triton-0.1.4-py3-none-any.whl size=28163 sha256=34

Looking in indexes: https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/Triton-Nightly/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 2.1 MB/s eta 0:00:00


In [ ]:
# Need to delete the local CUDA so jaxlib uses the pip-installed cuda12
!rm -rf /usr/local/cuda-11.8

In [ ]:
import jax
import jaxlib
print(jax.__version__)
print(jaxlib.__version__)

0.4.16.dev20230831
0.4.16.dev20230831


In [ ]:
from functools import partial

from jax.experimental import pallas as pl
from jax import lax
import jax.numpy as jnp
import numpy as np

In [ ]:
def add_vectors_kernel(x_ref, y_ref, o_ref):
  x, y = x_ref[...], y_ref[...]
  o_ref[...] = x + y

@jax.jit
def add_vectors(x: jax.Array, y: jax.Array) -> jax.Array:
  return pl.pallas_call(add_vectors_kernel,
                        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype)
                        )(x, y)
add_vectors(jnp.arange(8), jnp.arange(8))

Array([ 0,  2,  4,  6,  8, 10, 12, 14], dtype=int32)

In [ ]:
def matmul_kernel(x_ref, y_ref, z_ref, *, bk: int):
  m, n = x_ref.shape[0], y_ref.shape[1]
  k = y_ref.shape[0]
  acc = jnp.zeros((m, n), dtype=jnp.float32)
  def body(i, acc):
    x_k = pl.load(x_ref, (slice(None), pl.ds(i * bk, bk)))
    y_k = pl.load(y_ref, (pl.ds(i * bk, bk), slice(None)))
    return acc + x_k @ y_k
  z_ref[...] = lax.fori_loop(0, k // bk, body, acc).astype(z_ref.dtype)


def matmul(x: jax.Array, y: jax.Array, *, bm: int = 32, bn: int = 32, bk: int = 32,
           debug: bool = False, interpret: bool = False):
  m, n, k = x.shape[0], y.shape[1], y.shape[0]
  grid = (m // bm, n // bn)
  return pl.pallas_call(
    partial(matmul_kernel, bk=bk),
    out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
    grid=grid,
    in_specs=[
      pl.BlockSpec(lambda i, j: (i, 0), (bm, k)),
      pl.BlockSpec(lambda i, j: (0, j), (k, bn))
    ],
    out_specs=[
      pl.BlockSpec(lambda i, j: (i, j), (bm, bn))
    ],
    debug=debug, interpret=interpret,
  )(x, y)
k1, k2 = jax.random.split(jax.random.PRNGKey(0))
x = jax.random.normal(k1, (1024, 1024))
y = jax.random.normal(k2, (1024, 1024))
z = matmul(x, y)
np.testing.assert_allclose(z, x @ y, atol=3e-4)

### Dumping IRs

In [ ]:
matmul(x, y, debug=True)

{ lambda ; a:Ref{float32[32,1024]} b:Ref{float32[1024,32]} c:Ref{float32[32,32]}. let
    d:f32[32,32] = broadcast_in_dim[broadcast_dimensions=() shape=(32, 32)] 0.0
    _:i32[] e:f32[32,32] = scan[
      jaxpr={ lambda ; f:Ref{float32[32,1024]} g:Ref{float32[1024,32]} h:i32[] i:f32[32,32]. let
          j:i32[] = add h 1
          k:i32[] = mul h 32
          l:f32[32,32] <- f[:,k:k+32]
          m:i32[] = mul h 32
          n:f32[32,32] <- g[m:m+32,:]
          o:f32[32,32] = dot_general[dimension_numbers=(([1], [0]), ([], []))] l
            n
          p:f32[32,32] = add i o
        in (j, p) }
      length=32
      linear=(False, False, False, False)
      num_carry=2
      num_consts=2
      reverse=False
      unroll=1
    ] a b 0 d
    c[:,:] <- e
  in () }
GridMapping(grid=(32, 32), block_mappings=(BlockMapping(block_shape=(32, 1024), index_map_jaxpr={ lambda ; a:i32[] b:i32[]. let  in (a, 0) }), BlockMapping(block_shape=(1024, 32), index_map_jaxpr={ lambda ; a:i32[] b:i32[]. 

Array([[ -2.1177232,  16.311369 , -21.961555 , ...,  27.046679 ,
        -11.998793 , -53.40969  ],
       [ 26.41615  , -13.70285  ,  50.694294 , ..., -24.011066 ,
         18.311237 , -23.518991 ],
       [-21.33602  , -17.737497 ,  24.050615 , ..., -84.47401  ,
        -37.780605 , -23.439087 ],
       ...,
       [-34.074284 ,  24.666935 ,  33.72663  , ...,  31.877283 ,
         12.100911 ,  69.84239  ],
       [ 18.791811 ,  36.604214 , -36.526222 , ..., -94.81274  ,
         29.64589  ,   2.5705454],
       [ 36.4221   ,  13.942829 ,  31.152529 , ...,  -2.4010022,
         -7.49099  ,  62.230156 ]], dtype=float32)